In [1]:
import sys
import os
from pathlib import Path
import yaml

from src.core.parser import HiveScriptParser
from src.transformers.optimized_pyspark_transformer import OptimizedPySparkTransformer
from src.jinja.environment import render_template
from src.paths import *

In [7]:
# Test with the k2_bank DDL file
script_name = "raw_k2_bank"
# script_name = "raw_general_reference_lookup"
# script_name = "raw_lmskibb2_tbl_account"
# script_name = "com_t_mhbos_m_client"
datalake_type_subfolder = 'dml'
datalake_layer_subfolder = script_name.split("_")[0]

# sql_file_path = PROJECT_ROOT / "samples" / "input" / "ddl" / "raw" / f"{script_name}.sql"
sql_file_path = DATALAKE_SCRIPT_DIR / datalake_type_subfolder / datalake_layer_subfolder / f"{script_name}.sql"

output_file_path = PROJECT_ROOT / "samples" / "converted" / datalake_type_subfolder / datalake_layer_subfolder / f"{script_name}.py"

variable_path = VARIABLE_CONFIG_PATH


output_file_path = PROJECT_ROOT / "samples" / "converted" / datalake_type_subfolder / datalake_layer_subfolder / f"{script_name}.py"

In [8]:
print(f"[1] Parsing file {sql_file_path.name}...")
context = HiveScriptParser.parse_file(str(sql_file_path))

print("[2] Initializing Optimized Transformer...")
transformer = OptimizedPySparkTransformer(config_root=PROJECT_ROOT / "configs")
render_model = transformer.transform(context)

print("[3] Rendering Template (optimized_pyspark.jinja)...")
final_script = render_template(
    template_name="pyspark/optimized_pyspark.jinja",
    render_model=render_model
)

# 2. Load mapping configuration from YAML
output_file_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_file_path, 'w', encoding='utf-8') as f:
    f.write(final_script)

print("\n" + "=" * 50)
print("OPTIMIZED PYTHON FILE RESULT")
print("=" * 50 + "\n")
print(final_script)


[1] Parsing file raw_k2_bank.sql...
[2] Initializing Optimized Transformer...
Optimize for <class 'sqlglot.expressions.Drop'>
condition_match_result for <class 'sqlglot.expressions.Drop'>: [False], False
table_name: k2_bank_et, suffix: ('_et',), result: True
condition_match_result for <class 'sqlglot.expressions.Drop'>: [True, True], True
Optimize for <class 'sqlglot.expressions.Create'>
condition_match_result for <class 'sqlglot.expressions.Create'>: [False], False
table_name: k2_bank_et, suffix: ('_et',), result: True
condition_match_result for <class 'sqlglot.expressions.Create'>: [False, True], False
condition_match_result for <class 'sqlglot.expressions.Create'>: [True], True
Optimize for <class 'sqlglot.expressions.Alter'>
condition_match_result for <class 'sqlglot.expressions.Alter'>: [True], True
Optimize for <class 'sqlglot.expressions.Alter'>
condition_match_result for <class 'sqlglot.expressions.Alter'>: [True], True
Optimize for <class 'sqlglot.expressions.Insert'>
conditio

In [4]:

rule_file_path = PROJECT_ROOT / "configs" / "rules" / "optimizations" / "k2.yaml"

with open(rule_file_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)



# for rule in config["rules"]:
#     if not rule.get("enabled", False):
#         continue
#
#     if self._is_rule_triggered(rule, context):
#         action_type = rule.get("action", {}).get("type")
#         handler = action_handlers.get(action_type)
#
#         if handler:
#             # A rule was triggered and a handler exists, stop processing more rules.
#             return handler(rule, node, context)
#
#         # Stop at the first triggered rule, even if the action is unknown.
#         return None

for rule_name, rule_param in config["rules"].items():
    print(rule_param.get("enabled", False))

True
True
True
True
